# Validação leve da intermediate

Este notebook inspeciona a `intermediate` sem consultar o conteúdo das `views`.
Ele usa `manifest.json` e `information_schema` para mostrar inventário, granularidade, colunas e testes declarados.


In [1]:
import json
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)
DATABASE_PATH = PROJECT_ROOT / 'edu_impacto_nem_multimodal.duckdb'
MANIFEST_PATH = PROJECT_ROOT / 'target' / 'manifest.json'

if not DATABASE_PATH.exists():
    raise FileNotFoundError(f'Banco DuckDB não encontrado em {DATABASE_PATH}. Rode o dbt antes de executar este notebook.')
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'manifest.json não encontrado em {MANIFEST_PATH}. Rode o dbt antes de executar este notebook.')

con = duckdb.connect(str(DATABASE_PATH), read_only=True)
manifest = json.loads(MANIFEST_PATH.read_text())

DATABASE_PATH


PosixPath('/home/leovianaf/projetos/edu-impacto-nem-multimodal/edu_impacto_nem_multimodal.duckdb')

In [2]:
INTERMEDIATE_MODELS = [
    'int_censo_escola_ano_2019_2025',
    'int_censo_curso_tecnico_escola_ano_2023_2025',
    'int_municipio_ano_contexto',
    'int_municipio_ano_oferta_educacional',
    'int_municipio_ano_nem',
    'int_municipio_ano_tecnico',
    'int_municipio_ano_saeb',
    'int_municipio_ano_saeb_ciclo',
    'int_municipio_ano_painel_educacional',
]

EXPECTED_GRANULARITY = {
    'int_censo_escola_ano_2019_2025': 'escola/ano',
    'int_censo_curso_tecnico_escola_ano_2023_2025': 'escola/ano/curso',
    'int_municipio_ano_contexto': 'municipio/ano',
    'int_municipio_ano_oferta_educacional': 'municipio/ano',
    'int_municipio_ano_nem': 'municipio/ano',
    'int_municipio_ano_tecnico': 'municipio/ano',
    'int_municipio_ano_saeb': 'municipio/ano/ano_saeb',
    'int_municipio_ano_saeb_ciclo': 'municipio/ano/ano_saeb/ciclo',
    'int_municipio_ano_painel_educacional': 'municipio/ano',
}

KEY_COLUMNS = {
    'int_censo_escola_ano_2019_2025': ['ano', 'id_municipio', 'id_escola'],
    'int_censo_curso_tecnico_escola_ano_2023_2025': ['ano', 'id_municipio', 'id_escola', 'id_area_curso_profissional', 'co_curso_educ_profissional'],
    'int_municipio_ano_contexto': ['ano', 'id_municipio'],
    'int_municipio_ano_oferta_educacional': ['ano', 'id_municipio'],
    'int_municipio_ano_nem': ['ano', 'id_municipio'],
    'int_municipio_ano_tecnico': ['ano', 'id_municipio'],
    'int_municipio_ano_saeb': ['ano', 'id_municipio'],
    'int_municipio_ano_saeb_ciclo': ['ano', 'ano_saeb', 'id_municipio'],
    'int_municipio_ano_painel_educacional': ['ano', 'id_municipio'],
}

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).fetchdf()


def node(model_name: str):
    return manifest['nodes'][f'model.edu_impacto_nem_multimodal.{model_name}']


def test_names(model_name: str):
    uid = f'model.edu_impacto_nem_multimodal.{model_name}'
    return [t for t in manifest.get('child_map', {}).get(uid, []) if t.startswith('test.')]


def columns_df(model_name: str) -> pd.DataFrame:
    declared = []
    for col_name, meta in node(model_name).get('columns', {}).items():
        tests = [t.get('name') for t in meta.get('tests', []) if t.get('name')]
        declared.append({
            'column_name': col_name,
            'description': meta.get('description'),
            'not_null_declared': 'not_null' in tests,
            'accepted_values_declared': 'accepted_values' in tests,
            'relationship_declared': 'relationships' in tests,
        })
    declared_df = pd.DataFrame(declared)
    physical_df = q(f"""
        select column_name, data_type, is_nullable
        from information_schema.columns
        where table_schema = 'main'
          and table_name = '{model_name}'
        order by ordinal_position
    """)
    return declared_df.merge(physical_df, on='column_name', how='left')


def inventory_df() -> pd.DataFrame:
    rows = []
    for model in INTERMEDIATE_MODELS:
        rows.append({
            'model_name': model,
            'materialized': node(model).get('config', {}).get('materialized'),
            'granularity': EXPECTED_GRANULARITY.get(model),
            'key_columns': ', '.join(KEY_COLUMNS.get(model, [])),
            'declared_columns': len(node(model).get('columns', {})),
            'declared_tests': len(test_names(model)),
        })
    return pd.DataFrame(rows)


In [3]:
# Inventário estrutural da intermediate sem tocar nas views
inventory_df()


,model_name,materialized,granularity,key_columns,declared_columns,declared_tests
0,int_censo_escola_ano_2019_2025,view,escola/ano,"ano, id_municipio, id_escola",9,7
1,int_censo_curso_tecnico_escola_ano_2023_2025,table,escola/ano/curso,"ano, id_municipio, id_escola, id_area_curso_profissional, co_curso_educ_profissional",5,3
2,int_municipio_ano_contexto,table,municipio/ano,"ano, id_municipio",5,4
3,int_municipio_ano_oferta_educacional,view,municipio/ano,"ano, id_municipio",6,4
4,int_municipio_ano_nem,view,municipio/ano,"ano, id_municipio",5,3
5,int_municipio_ano_tecnico,view,municipio/ano,"ano, id_municipio",4,3
6,int_municipio_ano_saeb,table,municipio/ano/ano_saeb,"ano, id_municipio",5,3
7,int_municipio_ano_saeb_ciclo,table,municipio/ano/ano_saeb/ciclo,"ano, ano_saeb, id_municipio",5,4
8,int_municipio_ano_painel_educacional,table,municipio/ano,"ano, id_municipio",7,4


In [4]:
# Quais testes estão declarados por modelo
pd.DataFrame([
    {
        'model_name': model,
        'declared_tests': len(test_names(model)),
        'sample_tests': ', '.join(test_names(model)[:6])
    }
    for model in INTERMEDIATE_MODELS
])


,model_name,declared_tests,sample_tests
0,int_censo_escola_ano_2019_2025,7,test.edu_impacto_nem_multimodal.accepted_values_int_censo_escola_ano_2019_2025_fonte_censo__microdados_ed_basica_201...
1,int_censo_curso_tecnico_escola_ano_2023_2025,3,"test.edu_impacto_nem_multimodal.not_null_int_censo_curso_tecnico_escola_ano_2023_2025_ano.b04c18fdcb, test.edu_impac..."
2,int_municipio_ano_contexto,4,test.edu_impacto_nem_multimodal.accepted_values_int_municipio_ano_contexto_periodo_nem__pre_nem__implementacao_nem__...
3,int_municipio_ano_oferta_educacional,4,"test.edu_impacto_nem_multimodal.assert_int_municipio_aggregates_expected_municipio_coverage, test.edu_impacto_nem_mu..."
4,int_municipio_ano_nem,3,"test.edu_impacto_nem_multimodal.assert_int_municipio_aggregates_expected_municipio_coverage, test.edu_impacto_nem_mu..."
5,int_municipio_ano_tecnico,3,"test.edu_impacto_nem_multimodal.assert_int_municipio_aggregates_expected_municipio_coverage, test.edu_impacto_nem_mu..."
6,int_municipio_ano_saeb,3,"test.edu_impacto_nem_multimodal.accepted_values_int_municipio_ano_saeb_ano__2019__2021__2023.cda631cf1b, test.edu_im..."
7,int_municipio_ano_saeb_ciclo,4,"test.edu_impacto_nem_multimodal.accepted_values_int_municipio_ano_saeb_ciclo_ano_saeb__2019__2021__2023.9ceabdc123, ..."
8,int_municipio_ano_painel_educacional,4,test.edu_impacto_nem_multimodal.accepted_values_int_municipio_ano_painel_educacional_periodo_nem__pre_nem__implement...


In [5]:
# Colunas e nulabilidade declarada do painel consolidado, sem consulta aos dados
columns_df('int_municipio_ano_painel_educacional')


,column_name,description,not_null_declared,accepted_values_declared,relationship_declared,data_type,is_nullable
0,ano,Ano analítico do painel municipal consolidado.,False,False,False,INTEGER,YES
1,id_municipio,Código IBGE do município.,False,False,False,VARCHAR,YES
2,periodo_nem,Recorte temporal analítico relativo ao NEM.,False,False,False,VARCHAR,YES
3,ano_saeb_referencia,Ano da edição SAEB usada como referência no ciclo do painel.,False,False,False,INTEGER,YES
4,tem_saeb_no_ano_flag,Flag de disponibilidade do SAEB vigente no painel consolidado.,False,False,False,INTEGER,YES
5,prop_mat_em_tecnico_profissional,Proporção de matrículas de EM ligadas ao eixo técnico-profissional.,False,False,False,DOUBLE,YES
6,prop_escolas_com_curso_tecnico,Proporção de escolas com curso técnico no município/ano.,False,False,False,DOUBLE,YES


In [6]:
# Colunas e nulabilidade declarada do painel escola/ano compatibilizado
columns_df('int_censo_escola_ano_2019_2025')


,column_name,description,not_null_declared,accepted_values_declared,relationship_declared,data_type,is_nullable
0,ano,Ano censitário da observação escola/ano.,False,False,False,INTEGER,YES
1,id_municipio,Código IBGE do município da escola.,False,False,False,VARCHAR,YES
2,id_escola,Código da escola no Censo Escolar.,False,False,False,VARCHAR,YES
3,fonte_censo,Layout de origem usado para a harmonização da linha.,False,False,False,VARCHAR,YES
4,periodo_nem,Recorte temporal analítico relativo ao NEM.,False,False,False,VARCHAR,YES
5,in_med_compat,Indicador compatibilizado de oferta de ensino médio entre 2019 e 2025.,False,False,False,INTEGER,YES
6,in_prof_tec_compat,Indicador compatibilizado de presença de oferta técnico-profissional.,False,False,False,INTEGER,YES
7,qt_mat_med_compat,Quantidade compatível de matrículas do ensino médio.,False,False,False,INTEGER,YES
8,qt_mat_med_int_compat,Quantidade compatível de matrículas do ensino médio em tempo integral.,False,False,False,INTEGER,YES


In [7]:
# Colunas-chave que ajudam a validar a granularidade esperada
pd.DataFrame([
    {
        'model_name': model,
        'granularity': EXPECTED_GRANULARITY[model],
        'key_columns': ', '.join(KEY_COLUMNS[model])
    }
    for model in INTERMEDIATE_MODELS
])


,model_name,granularity,key_columns
0,int_censo_escola_ano_2019_2025,escola/ano,"ano, id_municipio, id_escola"
1,int_censo_curso_tecnico_escola_ano_2023_2025,escola/ano/curso,"ano, id_municipio, id_escola, id_area_curso_profissional, co_curso_educ_profissional"
2,int_municipio_ano_contexto,municipio/ano,"ano, id_municipio"
3,int_municipio_ano_oferta_educacional,municipio/ano,"ano, id_municipio"
4,int_municipio_ano_nem,municipio/ano,"ano, id_municipio"
5,int_municipio_ano_tecnico,municipio/ano,"ano, id_municipio"
6,int_municipio_ano_saeb,municipio/ano/ano_saeb,"ano, id_municipio"
7,int_municipio_ano_saeb_ciclo,municipio/ano/ano_saeb/ciclo,"ano, ano_saeb, id_municipio"
8,int_municipio_ano_painel_educacional,municipio/ano,"ano, id_municipio"


## Leitura prática

Este notebook garante rapidamente que:
- a `intermediate` existe e está materializada como esperado;
- as chaves e colunas críticas estão presentes;
- os testes declarados no dbt existem no manifesto.

Ele não substitui `dbt test`. Ele é um apoio visual para a camada já validada por testes do dbt.
